# Residência em IA — Semana 2: Base de Conhecimento e Recuperação (Primeiro Mecanismo RAG)

**Papel:** Engenheiro de IA Sênior & Mentor Técnico  
**Objetivo da Entrega:** Implementar um pipeline funcional, modular e auditável de **Retrieval-Augmented Generation (RAG)** voltado para orientação de estudos, produtividade e preparação por editais/trilhas autodidatas.

---

### Arquitetura do Pipeline RAG

Um sistema RAG conecta uma base de conhecimento confiável a um Grande Modelo de Linguagem (LLM) para reduzir alucinações e fundamentar (*grounding*) as respostas em fontes reais:

1. **Ingestão (`Ingestion`)**: Leitura de arquivos locais (`.pdf`, `.txt`) ou carregamento de fallback em memória com metadados rastreáveis.
2. **Segmentação (`Chunking`)**: Divisão dos documentos em blocos menores (`chunk_size=500`, `chunk_overlap=50`) preservando a coesão semântica.
3. **Vetorização e Persistência (`Embedding & Vector Store`)**: Transformação de texto em vetores densos e armazenamento local no **ChromaDB**.
4. **Recuperação (`Retrieval`)**: Busca por similaridade semântica (*Top-K = 3*) com inspeção de scores de distância/relevância.
5. **Geração Fundamentada (`Grounded Generation`)**: Construção da Chain RAG com prompt restritivo e avaliação prática com perguntas das personas do projeto.

## 1. Setup do Ambiente

> **Dica do Mentor:** Em projetos de produção e prototipação estruturada, sempre separamos a instalação de dependências da inicialização de configurações e chaves de API. Nunca exponha chaves diretamente no código-fonte (*hardcoding*); utilize arquivos `.env` gerenciados pelo `python-dotenv`.

In [ ]:
# Instalação silenciosa dos pacotes essenciais para o pipeline RAG
%pip install -q langchain langchain-core langchain-community langchain-text-splitters langchain-chroma langchain-openai langchain-google-genai chromadb pypdf tiktoken python-dotenv

In [ ]:
import logging
import os
from pathlib import Path
from typing import Any, Dict, List, Tuple

from dotenv import load_dotenv

# Configuração de logging limpo e informativo
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("RAGPipelineSemana2")


def setup_environment(env_path: str = ".env") -> str:
    """Carrega variáveis de ambiente e identifica o provedor de IA configurado.

    Suporta OPENAI_API_KEY (padrão) ou GOOGLE_API_KEY (Gemini) para máxima
    flexibilidade durante a Residência.
    """
    load_dotenv(dotenv_path=env_path, override=False)

    if os.getenv("OPENAI_API_KEY"):
        logger.info("Variável OPENAI_API_KEY detectada com sucesso.")
        return "openai"
    if os.getenv("GOOGLE_API_KEY"):
        logger.info("Variável GOOGLE_API_KEY detectada com sucesso.")
        return "google"

    logger.warning(
        "Nenhuma chave (OPENAI_API_KEY ou GOOGLE_API_KEY) encontrada no ambiente. "
        "Crie um arquivo .env na raiz do projeto com OPENAI_API_KEY='sua-chave' "
        "ou GOOGLE_API_KEY='sua-chave' antes de executar as etapas 4 e 5."
    )
    return "unconfigured"


ACTIVE_PROVIDER = setup_environment()

## 2. Ingestão da Base de Conhecimento

Nesta etapa, construímos uma camada de ingestão resiliente:
- Varre o diretório `./data/` em busca de arquivos `.txt` e `.pdf`.
- Trata exceções individualmente por arquivo (para que um PDF corrompido não interrompa toda a carga).
- Caso a pasta `./data/` não exista ou esteja vazia, aciona automaticamente um **fallback sintético em memória** rico em conteúdo sobre **técnicas de estudo, rotinas de produtividade, combate ao *overplanning* e estudo estratégico por editais**.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document


def get_synthetic_fallback_documents() -> List[Document]:
    """Retorna uma base de conhecimento sintética em memória caso ./data/ esteja vazia."""
    return [
        Document(
            page_content=(
                "GUIA DE CONCILIAÇÃO DE ROTINA E PRODUTIVIDADE SUSTENTÁVEL.\n"
                "Para estudantes e profissionais que trabalham em período integral (8h diárias), "
                "a consistência supera a intensidade eventual. A estratégia recomendada é o "
                "Timeboxing em Blocos Protegidos: reserve de 60 a 90 minutos pela manhã antes "
                "do expediente para o estudo de maior carga cognitiva (Teoria Complexa ou Prática "
                "Deliberada) e 30 a 45 minutos à noite exclusivamente para revisão ativa. "
                "Utilize micro-blocos de 25 minutos (Técnica Pomodoro adaptada) em janelas de "
                "deslocamento ou almoço para leitura de flashcards. Nunca sacrifique o sono "
                "abaixo de 7 horas, pois a consolidação da memória de longo prazo ocorre durante "
                "o sono REM e o sono profundo."
            ),
            metadata={
                "source": "fallback://guia_rotina_produtividade.txt",
                "topic": "conciliacao_rotina",
                "persona": "Profissional em Transição / Concurseiro Trabalhador",
            },
        ),
        Document(
            page_content=(
                "COMO EVITAR O OVERPLANNING (ARMADILHA DO EXCESSO DE PLANEJAMENTO).\n"
                "O overplanning é uma forma sofisticada de procrastinação em que o estudante "
                "gasta horas colorindo planilhas, trocando de aplicativos de notas e reorganizando "
                "cronogramas rígidos que quebram no primeiro imprevisto. Para combater o "
                "overplanning, aplique três regras práticas:\n"
                "1. Ciclo de Estudos Flexível em vez de Grade Horária Fixa: organize as matérias "
                "em uma fila sequencial de blocos de 1 hora; se imprevistos ocorrerem na terça-feira, "
                "basta retomar do exato ponto da fila na quarta-feira sem precisar refazer a planilha.\n"
                "2. Regra dos 10%: o tempo gasto planejando a semana nunca deve ultrapassar 10% "
                "do tempo líquido de estudo (ex.: no máximo 15 minutos no domingo).\n"
                "3. Gatilho de Ação Mínima Viável (Regra dos 5 Minutos): inicie a sessão resolvendo "
                "3 questões ou escrevendo um resumo livre antes de abrir qualquer ferramenta de gestão."
            ),
            metadata={
                "source": "fallback://manual_anti_overplanning.txt",
                "topic": "anti_overplanning",
                "persona": "Estudante Perfeccionista / Ansioso",
            },
        ),
        Document(
            page_content=(
                "METODOLOGIAS ATIVAS DE ESTUDO: ACTIVE RECALL E REPETIÇÃO ESPAÇADA.\n"
                "Estudos passivos como releitura linear, grifos excessivos e cópia de resumos "
                "geram a 'ilusão de competência'. Para retenção duradoura, priorize a Recordação "
                "Ativa (Active Recall): feche o material após 20 minutos de leitura e explique o "
                "conceito com suas próprias palavras (Técnica Feynman) ou resolva baterias de "
                "exercícios imediatamente. Combine isso com a Repetição Espaçada (intervalos "
                "progressivos de 1, 7, 16 e 35 dias) guiada pela taxa de acerto: tópicos com "
                "menos de 70% de acerto devem retornar à fila prioritária de revisão em 48 horas."
            ),
            metadata={
                "source": "fallback://metodologias_ativas_retencao.txt",
                "topic": "tecnicas_de_estudo",
                "persona": "Geral",
            },
        ),
        Document(
            page_content=(
                "PLANEJAMENTO AUTODIDATA E ESTRATÉGIA DE EDITAIS VERTICALIZADOS.\n"
                "No aprendizado autodidata e na preparação orientada por editais, o erro mais "
                "comum é estudar o conteúdo de forma linear do tópico 1 ao último com o mesmo peso. "
                "Utilize a Matriz de Priorização Peso x Domínio (Princípio de Pareto 80/20):\n"
                "- Verticalize o edital ou ementa técnica em uma lista atômica de subtópicos.\n"
                "- Classifique cada subtópico por Incidência/Relevância Prática (Alta, Média, Baixa) "
                "e Nível de Domínio Atual (Iniciante, Intermediário, Avançado).\n"
                "- Aloque 70% da carga horária nos tópicos de Alta Relevância e Baixo Domínio.\n"
                "- Para autodidatas em tecnologia e IA, aplique o Project-Based Learning (PBL): "
                "para cada módulo teórico estudado, construa um micro-artefato prático (notebook, "
                "script ou resumo executivo) validando o aprendizado em um caso real."
            ),
            metadata={
                "source": "fallback://estrategia_editais_autodidata.txt",
                "topic": "planejamento_autodidata_editais",
                "persona": "Autodidata / Candidato de Alta Performance",
            },
        ),
    ]


def load_knowledge_base(data_dir: str = "./data") -> List[Document]:
    """Carrega documentos .txt e .pdf da pasta especificada ou usa fallback em memória."""
    dir_path = Path(data_dir)
    loaded_docs: List[Document] = []

    if not dir_path.exists():
        logger.info(f"Diretório '{data_dir}' não encontrado. Criando diretório para uso futuro...")
        dir_path.mkdir(parents=True, exist_ok=True)

    supported_files = sorted(
        [f for f in dir_path.rglob("*") if f.is_file() and f.suffix.lower() in {".txt", ".pdf"}]
    )

    for file_path in supported_files:
        try:
            ext = file_path.suffix.lower()
            if ext == ".txt":
                loader = TextLoader(str(file_path), encoding="utf-8")
            else:
                loader = PyPDFLoader(str(file_path))

            docs = loader.load()
            loaded_docs.extend(docs)
            logger.info(f"Arquivo carregado: {file_path.name} ({len(docs)} página(s)/documento(s))")
        except Exception as exc:
            logger.error(f"Erro ao carregar o arquivo '{file_path}': {exc}")

    if not loaded_docs:
        logger.warning(
            f"Nenhum documento válido encontrado em '{data_dir}'. "
            "Ativando fallback com base de conhecimento sintética em memória."
        )
        loaded_docs = get_synthetic_fallback_documents()

    logger.info(f"Total de documentos brutos carregados na base: {len(loaded_docs)}")
    return loaded_docs


# Execução da ingestão
raw_documents = load_knowledge_base("./data")

## 3. Processamento de Texto e Chunking

> **Por que fazer Chunking?**  
> Modelos de Embedding têm limite de janela e representam melhor o significado quando o trecho foca em um único conceito ou parágrafo. O `RecursiveCharacterTextSplitter` tenta quebrar o texto seguindo uma hierarquia natural (`["\n\n", "\n", ". ", " ", ""]`), preservando parágrafos e frases completas sempre que possível.
>
> - `chunk_size=500`: Tamanho máximo (em caracteres) de cada fragmento.
> - `chunk_overlap=50`: Sobreposição de 10% entre fragmentos adjacentes para evitar que ideias sejam cortadas ao meio nas bordas.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Parâmetros configuráveis de segmentação
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50


def split_documents(
    documents: List[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> List[Document]:
    """Divide documentos em chunks menores preservando metadados e contexto nas bordas."""
    if not documents:
        raise ValueError("A lista de documentos para chunking está vazia.")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,  # Registra a posição inicial do chunk no documento original
    )

    chunks = text_splitter.split_documents(documents)

    # Enriquecimento de metadados com identificador sequencial do chunk
    for idx, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = idx

    return chunks


document_chunks = split_documents(raw_documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print("=" * 70)
print(f"RESUMO DO CHUNKING (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
print("=" * 70)
print(f"Documentos originais : {len(raw_documents)}")
print(f"Total de chunks      : {len(document_chunks)}")
print("-" * 70)
print("INSPEÇÃO DO PRIMEIRO CHUNK (document_chunks[0]):")
print(f"Tamanho (caracteres) : {len(document_chunks[0].page_content)}")
print(f"Metadados            : {document_chunks[0].metadata}")
print("\nConteúdo do Chunk #0:")
print(document_chunks[0].page_content)
print("=" * 70)

## 4. Vetorização e Vector Store (ChromaDB)

Agora convertemos cada chunk em um vetor numérico denso (*embedding*) que captura seu significado semântico e o armazenamos localmente no **ChromaDB** (`./chroma_db`).

> **Nota Técnica:** Incluímos suporte automático aos embeddings da **OpenAI** (`text-embedding-3-small`) e do **Google Gemini** (`models/text-embedding-004` / `gemini-embedding-001`), além de um *fallback determinístico local* (`FakeEmbeddings` / TF-IDF hash) caso você queira testar a mecânica completa do notebook antes de inserir sua chave de API no `.env`.

In [ ]:
import hashlib
import math
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings


class LocalKeywordHashEmbeddings(Embeddings):
    """Fallback local leve baseado em n-grams de palavras para execução offline sem chave de API."""

    def __init__(self, dim: int = 256) -> None:
        self.dim = dim

    def _embed(self, text: str) -> List[float]:
        vec = [0.0] * self.dim
        tokens = [t.strip(".,;:?!()[]{}\"'").lower() for t in text.split() if len(t) > 2]
        if not tokens:
            return [1.0 / math.sqrt(self.dim)] * self.dim
        for token in tokens:
            h = int(hashlib.md5(token.encode("utf-8")).hexdigest(), 16)
            idx = h % self.dim
            sign = 1.0 if ((h >> 8) & 1) == 0 else -1.0
            vec[idx] += sign
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        return [v / norm for v in vec]

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text)


def get_embedding_model(provider: str) -> Embeddings:
    """Inicializa o modelo de embeddings de acordo com o provedor configurado."""
    try:
        if provider == "openai":
            from langchain_openai import OpenAIEmbeddings

            logger.info("Inicializando OpenAIEmbeddings (model='text-embedding-3-small')...")
            return OpenAIEmbeddings(model="text-embedding-3-small")

        if provider == "google":
            from langchain_google_genai import GoogleGenerativeAIEmbeddings

            logger.info("Inicializando GoogleGenerativeAIEmbeddings (model='models/text-embedding-004')...")
            return GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
    except Exception as exc:
        logger.error(f"Erro ao inicializar provedor de embeddings '{provider}': {exc}")

    logger.warning(
        "Usando LocalKeywordHashEmbeddings (modo demonstração offline). "
        "Configure OPENAI_API_KEY ou GOOGLE_API_KEY no .env para embeddings neurais de produção."
    )
    return LocalKeywordHashEmbeddings(dim=256)


def build_or_update_vectorstore(
    chunks: List[Document],
    embedding_model: Embeddings,
    persist_directory: str = "./chroma_db",
    collection_name: str = "residencia_ia_semana2",
) -> Chroma:
    """Cria e persiste o banco vetorial local no ChromaDB de forma idempotente."""
    try:
        vectorstore = Chroma(
            collection_name=collection_name,
            embedding_function=embedding_model,
            persist_directory=persist_directory,
        )
        # Limpa registros anteriores da mesma coleção para evitar duplicatas em reexecuções do notebook
        existing_ids = vectorstore.get().get("ids", [])
        if existing_ids:
            vectorstore.delete(ids=existing_ids)

        vectorstore.add_documents(documents=chunks)
        logger.info(
            f"Vector Store ChromaDB persistido em '{persist_directory}' "
            f"com {len(chunks)} chunks indexados."
        )
        return vectorstore
    except Exception as exc:
        logger.error(f"Falha crítica ao criar/persistir o ChromaDB: {exc}")
        raise


# Inicialização dos embeddings e persistência no ChromaDB
embedding_model = get_embedding_model(ACTIVE_PROVIDER)
vectorstore = build_or_update_vectorstore(document_chunks, embedding_model)

# Demonstração de busca por similaridade semântica com scores (Top-K = 3)
TOP_K = 3
sample_query = "Como organizar minha rotina de estudos trabalhando 8 horas por dia?"

# similarity_search_with_score retorna tuplas (Document, score_de_distancia)
retrieved_with_scores: List[Tuple[Document, float]] = vectorstore.similarity_search_with_score(
    query=sample_query,
    k=TOP_K,
)

print("\n" + "=" * 70)
print(f"DEMONSTRAÇÃO DE BUSCA SEMÂNTICA (k={TOP_K})")
print(f"Consulta: '{sample_query}'")
print("=" * 70)

for rank, (doc, score) in enumerate(retrieved_with_scores, start=1):
    print(f"\n[Top #{rank}] Score (Distância L2 - menor é mais próximo): {score:.4f}")
    print(f"Fonte : {doc.metadata.get('source')} | Chunk ID: {doc.metadata.get('chunk_id')}")
    print(f"Trecho: {doc.page_content}")

## 5. Construção da Chain RAG e Geração

Na etapa final, conectamos o **Retriever** (`k=3`) ao **LLM** utilizando um template de prompt rigoroso que instrui o modelo a:
1. Responder **estritamente** com base nos fragmentos de contexto fornecidos.
2. Citar de qual fonte/documento a recomendação foi extraída.
3. Declarar explicitamente caso a informação não conste na base de conhecimento, evitando alucinações.

Em seguida, executamos **3 perguntas de teste** representando as dores reais das nossas personas:
- **Persona 1 (Rotina intensa):** Conciliação de trabalho em período integral e estudos diários.
- **Persona 2 (Procrastinação por perfeccionismo):** Técnicas práticas para superar o *overplanning*.
- **Persona 3 (Autodidata / Concurseiro):** Planejamento autodidata e priorização estratégica de editais.

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


class GroundedExtractiveFallbackLLM(BaseChatModel):
    """LLM de contingência determinístico caso nenhuma chave de API esteja configurada no .env.

    Sintetiza os trechos recuperados do prompt para que o pipeline RAG possa ser
    validado de ponta a ponta mesmo em ambientes sem chave ativa.
    """

    @property
    def _llm_type(self) -> str:
        return "grounded-extractive-fallback"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Any = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> ChatResult:
        full_prompt = "\n".join(str(m.content) for m in messages)
        context_section = ""
        if "CONTEXTO RECUPERADO:" in full_prompt and "PERGUNTA DO USUÁRIO:" in full_prompt:
            context_section = (
                full_prompt.split("CONTEXTO RECUPERADO:")[1]
                .split("PERGUNTA DO USUÁRIO:")[0]
                .strip()
            )

        response_text = (
            "[Modo Offline — Síntese Estritamente Baseada no Contexto Recuperado]\n"
            "Com base exclusivamente nos fragmentos recuperados da base de conhecimento, "
            "seguem as diretrizes recomendadas:\n\n"
            f"{context_section}\n\n"
            "(Configure OPENAI_API_KEY ou GOOGLE_API_KEY no arquivo .env para geração fluida via LLM neural.)"
        )
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=response_text))])


def get_llm(provider: str, temperature: float = 0.0) -> BaseChatModel:
    """Inicializa o modelo de linguagem (LLM) com temperatura baixa para máxima fidelidade factual."""
    try:
        if provider == "openai":
            from langchain_openai import ChatOpenAI

            logger.info("Inicializando ChatOpenAI (model='gpt-4o-mini', temperature=0.0)...")
            return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)

        if provider == "google":
            from langchain_google_genai import ChatGoogleGenerativeAI

            logger.info("Inicializando ChatGoogleGenerativeAI (model='gemini-2.5-flash', temperature=0.0)...")
            return ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=temperature)
    except Exception as exc:
        logger.error(f"Erro ao inicializar o LLM do provedor '{provider}': {exc}")

    logger.warning("Inicializando GroundedExtractiveFallbackLLM (modo offline sem chave de API).")
    return GroundedExtractiveFallbackLLM()


def format_retrieved_docs(docs: List[Document]) -> str:
    """Formata a lista de documentos recuperados com seus metadados para injeção no prompt."""
    formatted_parts = []
    for idx, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "desconhecida")
        chunk_id = doc.metadata.get("chunk_id", "N/A")
        formatted_parts.append(
            f"--- [Fragmento #{idx} | Fonte: {source} | Chunk ID: {chunk_id}] ---\n{doc.page_content}"
        )
    return "\n\n".join(formatted_parts)


# 1. Definição do Prompt Estruturado de QA com restrição estrita ao contexto
RAG_SYSTEM_PROMPT = """Você é um Mentor Técnico de Estudos e Produtividade altamente rigoroso.
Sua tarefa é responder à pergunta do estudante utilizando ESTRITAMENTE as informações presentes nos fragmentos do CONTEXTO RECUPERADO abaixo.

Regras obrigatórias:
1. Não invente informações, não extrapole além do texto e não utilize conhecimentos externos.
2. Se a resposta não estiver contida no contexto recuperado, responda exatamente: "Não encontrei informações suficientes na base de conhecimento para responder a essa pergunta."
3. Organize a resposta em tópicos claros e mencione a fonte ([Fonte: ...]) utilizada.
"""

RAG_USER_PROMPT = """CONTEXTO RECUPERADO:
{context}

PERGUNTA DO USUÁRIO:
{question}

RESPOSTA FUNDAMENTADA:"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

# 2. Criação do Retriever (k=3) e da Chain RAG em LCEL
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
llm = get_llm(ACTIVE_PROVIDER, temperature=0.0)

# RunnableParallel mantém tanto os documentos recuperados quanto a resposta final no dicionário de saída
rag_chain = RunnableParallel(
    {
        "context_docs": retriever,
        "question": RunnablePassthrough(),
    }
).assign(
    answer=(
        RunnablePassthrough.assign(
            context=RunnableLambda(lambda x: format_retrieved_docs(x["context_docs"]))
        )
        | qa_prompt
        | llm
        | StrOutputParser()
    )
)


def execute_rag_query(question: str, persona_label: str) -> Dict[str, Any]:
    """Executa uma consulta no pipeline RAG e exibe Pergunta, Contexto e Resposta formatados."""
    try:
        result = rag_chain.invoke(question)
        context_docs: List[Document] = result["context_docs"]
        answer: str = result["answer"]

        print("\n" + "#" * 80)
        print(f"CENÁRIO DE TESTE: {persona_label}")
        print("#" * 80)
        print(f"\n[1] PERGUNTA:\n{question}")

        print(f"\n[2] CONTEXTO RECUPERADO ({len(context_docs)} chunks):")
        for idx, doc in enumerate(context_docs, start=1):
            source = doc.metadata.get("source", "N/A")
            chunk_id = doc.metadata.get("chunk_id", "N/A")
            topic = doc.metadata.get("topic", "geral")
            print(f"  -> Chunk #{idx} (id={chunk_id} | fonte={source} | tópico={topic}):")
            snippet = doc.page_content.replace("\n", " ")
            print(f"     \"{snippet}\"")

        print(f"\n[3] RESPOSTA FINAL GERADA:\n{answer}")
        print("#" * 80)
        return result
    except Exception as exc:
        logger.error(f"Erro durante a execução da consulta RAG: {exc}")
        return {"question": question, "context_docs": [], "answer": f"Erro: {exc}"}


# 3. Execução das 3 Perguntas de Teste simulando as Personas
test_scenarios = [
    (
        "Persona 1 — Conciliação de Rotina (Trabalho 8h + Estudos)",
        "Trabalho 8 horas por dia e sinto que não consigo render nos estudos. Como devo dividir meus blocos de estudo diários e cuidar do sono?",
    ),
    (
        "Persona 2 — Combate ao Overplanning",
        "Eu perco horas montando planilhas e cronogramas que sempre atrasam quando surge um imprevisto. Quais técnicas práticas posso aplicar para evitar o overplanning?",
    ),
    (
        "Persona 3 — Planejamento Autodidata e Editais",
        "Como devo priorizar os tópicos de um edital extenso ou trilha autodidata em vez de estudar tudo de forma linear, e como validar meu aprendizado prático?",
    ),
]

rag_results = [
    execute_rag_query(question=q, persona_label=label)
    for label, q in test_scenarios
]